# Streamflow quality check

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from tqdm import tqdm
import os

# quality control
from saqc import SaQC
from plotly.subplots import make_subplots
from ipywidgets import interact, Dropdown
import plotly.graph_objects as go

from utils.config import find_repo_root, get

wd = find_repo_root()
os.chdir(wd)

## Data

In [ ]:
AndeanGC_data     = pd.read_csv('dataset/AndeanGC_data_1950_2024.csv', index_col=0, parse_dates=True)
AndeanGC_metadata = pd.read_csv('dataset/AndeanGC_metadata.csv', index_col=0)
AndeanGC_shape    = gpd.read_file('dataset/AndeanGC_shape.gpkg').set_index('gauge_id')

## Automatic quality check

In [ ]:
qc = SaQC(data=AndeanGC_data, scheme="dmp")

# perform quality checks
for column in tqdm(AndeanGC_data.columns):
    qc = (qc
          # Flag constant values (e.g., sensor stuck)
          .flagConstants(column, thresh=get('qc_constant_threshold'), window=get('qc_window'), min_periods=30)
          
          # Flag values outside reasonable range (adjust max)
          .flagRange(column, min=0, max=AndeanGC_data[column].quantile(0.99) * 6)

          # Flag isolates values
          .flagIsolated(column, gap_window=get('qc_window'), group_window="10D")

          # Flag using residuals TODO
          #.flagByScatterLowpass(column, window="5D", thresh=1)
    )

# Get flags dataframe and filter for quality_comment columns
flags_df = qc.flags.to_dataframe()
flags = [col for col in flags_df.columns if 'quality_comment' in col]
flags_df = flags_df[flags]
flags_df.columns = [col.replace('_quality_comment', '') for col in flags_df.columns]

for gauge in flags_df.columns:
    flags_df[gauge] = flags_df[gauge].apply(
        lambda x: eval(x) if isinstance(x, str) and x.strip() else {}
    ).apply(
        lambda x: x.get('test', '') if isinstance(x, dict) and x else ''
    )
flags_df = flags_df.replace('', np.nan)

In [ ]:
# Create summary dataframe and apply qc
summary_df = pd.DataFrame({
    'total_flags': flags_df.notna().sum(),
    'flagConstant': (flags_df == 'flagConstants').sum(),
    'flagIsolated': (flags_df == 'flagIsolated').sum(),
    'flagRange': (flags_df == 'flagRange').sum(),
})

AndeanGC_data_qc = AndeanGC_data[flags_df.isna()]
AndeanGC_metadata["days_w_data_qc"] = AndeanGC_data_qc.notna().sum()

## Manual quality check

In [ ]:
from ipywidgets import HBox, Label

station_dropdown = Dropdown(
    options=AndeanGC_data.columns.tolist(),
    description='Station:',
)

station_label = Label(value=station_dropdown.value)

def update_label(change):
    station_label.value = change['new']

station_dropdown.observe(update_label, names='value')

def plot_comparison(station):
    fig = make_subplots(rows=1, cols=1,
                        subplot_titles=[f'Streamflow Comparison - {station}'])
    fig.add_trace(go.Scatter(x=AndeanGC_data.index,
                             y=AndeanGC_data[station],
                             mode='lines',
                             name='Raw Data',
                             line=dict(color='blue', width=1),
                             opacity=0.7))
    flag_types = {'flagConstants': 'orange',
                  'flagIsolated': 'red',
                  'flagRange': 'purple'}
    for flag_type, color in flag_types.items():
        flagged_mask = (flags_df[station] == flag_type)
        if flagged_mask.any():
            fig.add_trace(go.Scatter(x=AndeanGC_data.index[flagged_mask],
                                     y=AndeanGC_data.loc[flagged_mask, station],
                                     mode='markers',
                                     name=flag_type,
                                     marker=dict(color=color, size=7)))
    fig.update_layout(title=f'Raw Data with Flagged Outliers - {station}',
                      xaxis_title='Date',
                      yaxis_title='Streamflow',
                      hovermode='x unified',
                      height=800, width=1200)
    valid_data = AndeanGC_data[station].dropna()
    if not valid_data.empty:
        fig.update_xaxes(range=[valid_data.index.min(), valid_data.index.max()])
    fig.show()

display(HBox([station_dropdown, station_label]))
interact(plot_comparison, station=station_dropdown)


## Save final dataset after QC

In [ ]:
to_remove = get('qc_stations_to_remove')
AndeanGC_metadata = AndeanGC_metadata[~AndeanGC_metadata.index.isin(to_remove)]
AndeanGC_metadata.to_csv('dataset/AndeanGC_metadata.csv')

AndeanGC_data_qc = AndeanGC_data_qc[AndeanGC_metadata.index]
AndeanGC_data_qc.to_csv('dataset/AndeanGC_data_1950_2024_qc.csv')

AndeanGC_shape = AndeanGC_shape.loc[AndeanGC_metadata.index]
AndeanGC_shape[["days_w_data_qc", "days_w_data"]] = AndeanGC_metadata[["days_w_data_qc", "days_w_data"]] 
AndeanGC_shape.to_file('dataset/AndeanGC_shape.gpkg', driver='GPKG')